In [1]:
# Parameters
BATCH_MODE = "true"

# Complexity-03i — L'exposant logarithmique de la FFT : des portes en $(\log N)^{1/2+o(1)}$

**Série Complexity — accrétion de 03** · Public : **Recherche** · Hommage : **James Cooley & John Tukey** (*An algorithm for the machine calculation of complex Fourier series*, 1965) et **Jacques Morgenstern** (*Note on a lower bound on the linear complexity of the fast Fourier transform*, 1973) — deux travaux cités d'après la bibliographie du préprint, dont ce carnet compte le circuit de l'un et rejoue l'argument de l'autre ; **OpenAI** (*The logarithmic exponent of finite Fourier transforms*, 24 novembre 2026) pour l'encadrement qui, dans un modèle plus général, les remplace.

**Navigation** : [Index de la série](README.md) | [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) | [03b — Hartmanis & Stearns](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) | [03c — Zoo et oracles](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) | [03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) | [03e — barrières d'exposants, word RAM](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) | [03f — la chaîne des réductions](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb) | [07c — le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb)

> **Bandeau d'honnêteté.** Ce carnet **lit** un préprint daté du 24 novembre 2026 ; il n'implémente **aucun** de ses circuits, et le préprint ne donne aucune constante explicite pour sa borne supérieure. Ce qui est mesuré ici est autre chose, et chaque cellule dit quoi : le **compte de portes** d'un réseau FFT écrit à la main pour de petites tailles, l'argument de **Morgenstern** rejoué en arithmétique exacte, et la **forme numérique** de l'écart entre les deux bornes. Le préprint est auto-publié et n'a pas été relu ; son statut de formalisation est relevé au README du dossier qui le porte, écrit tel que trouvé. Aucun `lake build` n'est lancé ici.

## 0. Pourquoi ce carnet

Le 24 novembre 2026, la même journée a vu paraître deux préprints de la même famille : celui qui, d'après le carnet 03h (en revue sous #28554), **ferme** l'exposant du produit matriciel ($\omega = 9/4$), et celui-ci, qui **encadre** le coût de la transformée de Fourier. Les deux ont la même forme — une borne inférieure et une borne supérieure **au même exposant logarithmique**, sur le modèle de circuit le plus permissif qu'on sache écrire — et c'est cette forme que la série n'avait pas encore rencontrée.

Le [03e](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) a lu la formalisation de speedups d'octobre ; le [03f](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb) a marché la chaîne des réductions qui les propage ; le [03d](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) a lu une réfutation annoncée. Tous décrivaient une **amélioration** : une borne supérieure meilleure, parfois une borne inférieure qui n'avait pas suivi. Ce carnet-ci lit un **encadrement** : la borne inférieure rejoint la borne supérieure en exposant, et ce qui était un algorithme devient un **exposant**. Le 03g (en revue sous #28573) tient, lui, les trois **barrières de preuve** — relativisation, preuves naturelles, algébrisation.

Le résultat, en une phrase : pour une transformée de Fourier sur un groupe abélien fini d'ordre $N$, le nombre minimal de **portes linéaires binaires** par échantillon est $(\log N)^{1/2+o(1)}$ — ni $\log N$ (la FFT de 1965), ni une constante. La FFT est donc **dépassable** (des circuits font mieux en exposant) mais **pas de beaucoup** (le gain par échantillon est au plus d'un facteur $\sqrt{\log N}$, à la constante $c$ près) : c'est l'objet de l'issue #28733.

Ce carnet fait trois choses, et rien de plus : (1) poser le **modèle** exactement, parce que tout le résultat vit dedans et qu'un compte de portes n'est pas un temps ; (2) **compter** les portes de la FFT à la main, exactement, puis rejouer l'argument de Morgenstern et montrer pourquoi il ne franchit pas le modèle à scalaires bornés ; (3) **calculer la forme de l'écart** — pourquoi il est asymptotique, et pourquoi aucune taille où il se réalise ne peut être nommée tant que la constante $C$ du théorème n'est pas connue — avant de dire ce qui reste ouvert.

In [2]:
import math
import time

import numpy as np

SEED = 20261124  # tout le carnet est rejouable a l'identique
rng = np.random.default_rng(SEED)
print(f"Imports OK : math, time, numpy {np.__version__}. SEED = {SEED}.")

Imports OK : math, time, numpy 2.4.2. SEED = 20261124.


## 1. Le modèle, mot pour mot

Tout le résultat vit dans le modèle, et le préprint le définit en quatre phrases (§1, p. 1 ; copie archivée au gisement, lue le 26 novembre 2026) :

> « Write $C_K(T)$ for the smallest number of gates in an acyclic linear circuit computing a matrix $T$ over a field $K$. Each gate forms $au + bv$ from two available values, with arbitrary $a, b \in K$. An output may select and scale a single input or gate value, or return zero. All logarithms in the paper are to base two. »

En clair : **une porte** est une somme pondérée de **deux** valeurs déjà disponibles, avec des coefficients **quelconques** du corps ; **rien d'autre n'est facturé**. Tout ce qui suit en découle.

| Objet | Coût dans le modèle |
|---|---|
| une porte $au + bv$, $a$ et $b$ quelconques du corps | **1** |
| multiplier par une racine de l'unité, par un twiddle, par n'importe quel scalaire | **0** (le scalaire est absorbé par la porte) |
| lire un fil, choisir un fil comme sortie, le multiplier pour la sortie | **0** |
| calculer un indice, préparer une racine de l'unité, allouer de la mémoire | **0** |
| une porte, traduite en opérations du corps | au plus **3** : « three field operations suffice for a weighted linear gate » |

Le théorème, ensuite (Théorème 1.1) : pour $J \ge 1$ transformées de groupes abéliens finis $G_1,\dots,G_J$ d'ordre commun $N \ge 2$, $N$ inversible dans $K$ et $K$ contenant une racine primitive d'ordre $\exp(G_j)$, la transformée $T$ étant la somme directe de ces blocs, chacun choisissant librement la transformée directe ou inverse,

$$c\, J\, N\sqrt{\log N} \;\le\; C_K(T) \;\le\; C\, J\, N\,(1 + \log N)^{1/2+\varepsilon}$$

où $c > 0$ est une constante absolue et $C$ ne dépend que de $\varepsilon > 0$. Le préprint écrit lui-même la limite : « Theorem 1.1 leaves open an upper bound with $\varepsilon = 0$. »

Trois précisions que le modèle impose, et qu'on oublie facilement :

1. **La borne ne facture que les portes.** Le préprint le dit à la fin du §1.2 : « The circuit size in Theorem 1.1 charges only for the binary linear gates. » L'un des algorithmes qu'il cite, celui de [34, Théorème 1.1], facture à l'inverse l'indexation des tableaux et la préparation des scalaires. Comparer ce chiffre à un temps d'exécution serait une faute de modèle ; ce carnet ne la commet pas, et la section 4 mesure même à quel point un temps ne suit pas un compte de portes.
2. **Ce qui est gratuit change tout.** Une multiplication par un twiddle ne coûte rien : le papillon complexe et le papillon entier ont donc le **même** prix. Un compte de portes n'est pas un compte d'opérations arithmétiques.
3. **Le corps compte.** Sur $\mathbb{C}$ toutes les racines existent ; sur un corps fini il faut la racine primitive et $N$ inversible. Le résultat vaut sur tout corps qui satisfait ces deux conditions — l'abstract : « This holds over every field in which $N$ is invertible and which contains the required roots of unity. »

Le préprint avertit enfin, au §1.3, qu'« over $\mathbb{Q}$ the map that adds the sum of all inputs to each input is dense and invertible, yet has a linear-size circuit ». Autrement dit : **le nombre d'entrées non nulles d'une matrice ne dit rien de son coût**. La section 2 en fait une mesure.

## 2. Compter les portes de la FFT, exactement

Le réseau papillon radix-2 traite $N = 2^k$ entrées en $k$ étages de $N/2$ papillons. Chaque papillon $(u, v) \mapsto (u+v,\; u-v)$ est **deux** portes au sens du préprint : l'une de coefficients $(1, 1)$, l'autre de coefficients $(1, -1)$ ; et les sorties du dernier étage **sont** les sorties, que sélectionner est gratuit. Le compte attendu est donc exactement $N\log_2 N$, sans terme d'ordre inférieur.

La cellule suivante écrit ce réseau comme un **programme de portes** — une liste de quadruplets $(a, i, b, j)$ — puis l'exécute avec un compteur. Trois contrôles l'accompagnent :

- le compte mesuré égale $N\log_2 N$, et se compare au compte naïf $N(N-1)$ d'une multiplication matrice-vecteur dense ;
- la valeur est vérifiée sur une entrée entière aléatoire, en arithmétique exacte, contre le produit matriciel direct ;
- la **matrice** entière est reconstruite — le programme est rejoué sur les $N$ vecteurs de la base — et doit égaler la matrice de Hadamard. C'est ce troisième contrôle qui interdit de compter un circuit qui ne calcule pas la bonne transformée.

In [3]:
def hadamard(N):
    """La matrice de Hadamard H_N, en entiers (H_1 = [1], H_2n = [[H, H], [H, -H]])."""
    H = np.array([[1]], dtype=np.int64)
    while H.shape[0] < N:
        H = np.block([[H, H], [H, -H]])
    return H


def programme_wht(N):
    """Le reseau papillon radix-2 comme programme de portes.
    Une porte est un quadruplet (a, i, b, j) : elle calcule a * fils[i] + b * fils[j].
    Les fils 0..N-1 sont les entrees ; chaque porte ajoute un fil.
    Rend (programme, position) : position[p] est le fil qui porte la sortie p."""
    assert N >= 2 and (N & (N - 1)) == 0
    prog = []
    position = list(range(N))
    taille = 2
    while taille <= N:
        demi = taille // 2
        for base in range(0, N, taille):
            for j in range(demi):
                i1, i2 = position[base + j], position[base + j + demi]
                q1 = N + len(prog)
                prog.append((1, i1, 1, i2))     # u + v
                q2 = N + len(prog)
                prog.append((1, i1, -1, i2))    # u - v
                position[base + j], position[base + j + demi] = q1, q2
        taille *= 2
    return prog, position


def evaluer(programme, x):
    """Execute le programme sur l'entree x. Rend (liste des fils, nombre de portes)."""
    fils = list(x)
    for (a, i, b, j) in programme:
        assert 0 <= i < len(fils) and 0 <= j < len(fils), "une porte lit un fil futur"
        fils.append(a * fils[i] + b * fils[j])
    return fils, len(programme)


def matrice_du_programme(programme, N, position):
    """Rejoue le programme sur chaque vecteur de la base : la matrice calculee."""
    fils = [np.eye(N, dtype=np.int64)[r] for r in range(N)]
    for (a, i, b, j) in programme:
        fils.append(a * fils[i] + b * fils[j])
    return np.array([fils[p] for p in position], dtype=np.int64)


print("Reseau radix-2 de la transformee de Walsh-Hadamard, compte en portes du modele")
print()
print("      N    portes    N*log2(N)    naif N*(N-1)    rapport    matrice = H_N    valeur exacte")
for k in range(1, 11):
    N = 1 << k
    prog, position = programme_wht(N)
    x = rng.integers(-9, 10, size=N)
    fils, n_portes = evaluer(prog, x)
    H = hadamard(N)
    matrice_ok = bool(np.array_equal(matrice_du_programme(prog, N, position), H))
    valeur_ok = [int(fils[p]) for p in position] == [int(v) for v in (H @ x)]
    print(f"{N:7d} {n_portes:9d} {N * k:12d} {N * (N - 1):15d} {N * (N - 1) / n_portes:9.1f} "
          f"{str(matrice_ok):>16s} {str(valeur_ok):>16s}")

Reseau radix-2 de la transformee de Walsh-Hadamard, compte en portes du modele

      N    portes    N*log2(N)    naif N*(N-1)    rapport    matrice = H_N    valeur exacte
      2         2            2               2       1.0             True             True
      4         8            8              12       1.5             True             True
      8        24           24              56       2.3             True             True
     16        64           64             240       3.8             True             True
     32       160          160             992       6.2             True             True
     64       384          384            4032      10.5             True             True
    128       896          896           16256      18.1             True             True
    256      2048         2048           65280      31.9             True             True


    512      4608         4608          261632      56.8             True             True


   1024     10240        10240         1047552     102.3             True             True


### Lecture du résultat — $N\log_2 N$ portes, exactement

Le compte mesuré **égale** $N\log_2 N$ sur toute la plage ($2$ portes à $N = 2$, $10\,240$ à $N = 1\,024$), et la matrice reconstruite est **exactement** celle de Hadamard, sans permutation ni changement de signe. La colonne du calcul naïf montre l'écart avec une multiplication dense : à $N = 1\,024$, $1\,047\,552$ portes contre $10\,240$, soit un facteur $102{,}3$ — l'écart entre le produit matrice-vecteur et la FFT, compté ici dans le modèle du préprint, où seule la porte est facturée.

Deux remarques de méthode. D'abord, le compte ne dépend pas des valeurs : le programme est le même pour tout $x$, et le vérifier sur une entrée suffit à vérifier le compteur, pas la correction — d'où le troisième contrôle, qui porte sur la matrice. Ensuite, la reconstruction matricielle coûte $N^2$ coefficients — $1\,024 \times 1\,024$ à la dernière ligne — et la cellule s'arrête à cette taille ; le **compteur**, lui, ne coûte rien, et le réseau se poursuit au-delà sans rien changer à la formule.

In [4]:
def renverser_bits(i, k):
    """Ecriture de i sur k bits, lue a l'envers."""
    r = 0
    for _ in range(k):
        r = (r << 1) | (i & 1)
        i >>= 1
    return r


def programme_dft(N):
    """Meme reseau, papillon complexe (u + w*v, u - w*v) : la racine de l'unite entre
    dans le coefficient de la porte, donc le compte de portes ne bouge pas.
    L'entree est lue dans l'ordre renverse par bits -- lire une entree ne coute rien
    dans le modele, et c'est cet ordre qui fait du reseau la vraie transformee."""
    k = N.bit_length() - 1
    w = np.exp(-2j * np.pi * np.arange(N) / N)
    prog = []
    position = [renverser_bits(p, k) for p in range(N)]
    taille = 2
    while taille <= N:
        demi = taille // 2
        for base in range(0, N, taille):
            for j in range(demi):
                i1, i2 = position[base + j], position[base + j + demi]
                twiddle = w[(j * (N // taille)) % N]
                q1 = N + len(prog)
                prog.append((1, i1, twiddle, i2))
                q2 = N + len(prog)
                prog.append((1, i1, -twiddle, i2))
                position[base + j], position[base + j + demi] = q1, q2
        taille *= 2
    return prog, position


def determinant_bareiss(M):
    """Determinant exact d'une matrice d'entiers, sans fractions (algorithme de Bareiss)."""
    A = [list(map(int, row)) for row in M]
    n = len(A)
    signe, precedent = 1, 1
    for k in range(n - 1):
        if A[k][k] == 0:
            for r in range(k + 1, n):
                if A[r][k] != 0:
                    A[k], A[r] = A[r], A[k]
                    signe = -signe
                    break
            else:
                return 0
        for i in range(k + 1, n):
            for j in range(k + 1, n):
                A[i][j] = (A[i][j] * A[k][k] - A[i][k] * A[k][j]) // precedent
        precedent = A[k][k]
        for i in range(k + 1, n):
            A[i][k] = 0
    return signe * A[n - 1][n - 1]


def programme_dense_creux(N):
    """T = I + J, ou J est remplie de 1 : la somme de toutes les entrees (N-1 portes),
    puis chaque sortie x_i + s (N portes). Rend (programme, fils de sortie)."""
    prog = []
    accumulateur = 0
    for i in range(1, N):
        q = N + len(prog)
        prog.append((1, accumulateur, 1, i))
        accumulateur = q
    sorties = []
    for i in range(N):
        q = N + len(prog)
        prog.append((1, i, 1, accumulateur))
        sorties.append(q)
    return prog, sorties


print("Transformee de Fourier discrete complexe, comparee a numpy.fft.fft")
print()
print("      N    portes    N*log2(N)    erreur relative max")
for k in [3, 4, 6, 8, 10]:
    N = 1 << k
    prog, position = programme_dft(N)
    x = rng.standard_normal(N) + 1j * rng.standard_normal(N)
    fils, n_portes = evaluer(prog, x)
    obtenu = np.array([fils[p] for p in position])
    reference = np.fft.fft(x)
    erreur = np.max(np.abs(obtenu - reference)) / np.max(np.abs(reference))
    print(f"{N:7d} {n_portes:9d} {N * k:12d} {erreur:22.1e}")

# la matrice du reseau, a N = 8, contre la matrice de Fourier elle-meme
N = 8
prog, position = programme_dft(N)
fils = [np.eye(N, dtype=complex)[r] for r in range(N)]
for (a, i, b, j) in prog:
    fils.append(a * fils[i] + b * fils[j])
M = np.array([fils[p] for p in position])
D = np.exp(-2j * np.pi * np.outer(np.arange(N), np.arange(N)) / N)
print()
print(f"A N = {N}, la matrice du programme est celle de Fourier : {np.allclose(M, D)}")

print()
print("Une matrice dense et bon marche : T = I + J (J entierement remplie de 1)")
print()
print("      N    portes    2N-1    naif N*(N-1)    det(T) = N+1    valeur exacte")
for N in [2, 4, 8, 16, 32]:
    prog, sorties = programme_dense_creux(N)
    x = rng.integers(-9, 10, size=N)
    fils, n_portes = evaluer(prog, x)
    T = np.eye(N, dtype=np.int64) + np.ones((N, N), dtype=np.int64)
    exact = [int(fils[p]) for p in sorties] == [int(v) for v in (T @ x)]
    print(f"{N:7d} {n_portes:9d} {2 * N - 1:7d} {N * (N - 1):15d} {determinant_bareiss(T):16d} {str(exact):>16s}")

Transformee de Fourier discrete complexe, comparee a numpy.fft.fft

      N    portes    N*log2(N)    erreur relative max
      8        24           24                6.6e-17
     16        64           64                1.4e-16
     64       384          384                3.1e-16
    256      2048         2048                5.9e-16
   1024     10240        10240                6.2e-16

A N = 8, la matrice du programme est celle de Fourier : True

Une matrice dense et bon marche : T = I + J (J entierement remplie de 1)

      N    portes    2N-1    naif N*(N-1)    det(T) = N+1    valeur exacte
      2         3       3               2                3             True
      4         7       7              12                5             True
      8        15      15              56                9             True
     16        31      31             240               17             True
     32        63      63             992               33             True


### Lecture du résultat — le twiddle est gratuit, et une matrice dense peut être linéaire

Le papillon complexe coûte exactement le même nombre de portes que le papillon entier — $N\log_2 N$ encore — et la matrice reconstruite à $N = 8$ est celle de Fourier, à l'erreur d'arrondi près. L'erreur relative maximum, lue dans la sortie de la cellule, reste au niveau du bruit de virgule flottante (de l'ordre de $10^{-16}$) : le réseau est correct, et son prix ne dépend pas du fait que les coefficients soient des racines de l'unité. **Dans ce modèle, une multiplication par un twiddle est gratuite** ; c'est pourquoi un compte de portes, dans ce modèle, ne se compare pas tel quel à un compte d'opérations arithmétiques.

Le second tableau répond à l'avertissement du §1.3 par une mesure : $T = I + J$, où $J$ est la matrice entièrement remplie de 1, est **dense** (aucun coefficient nul) et **inversible** (déterminant $N+1$, exactement, calculé par l'algorithme de Bareiss sans fractions) — et elle se calcule en $2N-1$ portes : une somme totale en $N-1$ portes, puis $N$ additions. À $N = 32$, $63$ portes contre $992$ pour le calcul naïf. Le nombre de coefficients non nuls d'une matrice ne dit donc rien de son coût, et c'est pourquoi la borne inférieure du préprint ne peut pas venir d'un simple argument de densité : il lui faut un argument qui combine rang et support (cité en section 5).

## 3. Morgenstern rejoué : pourquoi l'argument de volume s'arrête au modèle à scalaires bornés

L'ancienne borne inférieure est un argument de **volume**. Le préprint le résume au §1.2 (p. 3) en quatre phrases, citées ici telles quelles :

> « For the unnormalized complex transform, the determinant has modulus $N^{N/2}$. Morgenstern showed that if scalar multipliers have modulus at most one, an addition can at most double the largest determinant modulus attainable from the available forms. At least $\frac{1}{2}N\log_2 N$ additions are therefore necessary [33]. Normalizing the transform makes its determinant have modulus one. »

Trois ingrédients : (1) la transformée non normalisée a un déterminant de module $N^{N/2}$ ; (2) si les coefficients sont bornés par 1 en module, **une addition double au plus** le plus grand module de déterminant atteignable ; (3) il faut donc au moins $\frac{1}{2}N\log_2 N$ additions. La FFT radix-2, elle, en compte $N\log_2 N$ — deux par papillon : le double de ce que la borne exige, si bien que, dans ce modèle, la borne de Morgenstern laisse à la FFT un facteur 2 et pas plus.

La cellule suivante rejoue l'argument en arithmétique exacte : elle suit la famille des $N$ formes calculées, part de l'identité, applique les papillons dans l'ordre du réseau, et vérifie que le déterminant suivi vaut à l'arrivée, signe compris, celui que l'algorithme de Bareiss calcule **indépendamment** sur la matrice finale — la matrice de Hadamard, retrouvée telle quelle. Puis elle **casse** l'argument en changeant un seul jeu de coefficients, sans changer une seule porte : c'est ce que le préprint ajoute, deux phrases plus loin — « Theorem 1.1 permits arbitrary scalar coefficients, intermediate storage and reuse. »

In [5]:
def rejouer_morgenstern(N, scalaires=(1, 1)):
    """Suit la famille des N formes a travers le reseau papillon. Chaque papillon
    remplace (u, v) par (a*u + b*v, a*u - b*v) : le determinant de la famille est
    multiplie par det[[a, b], [a, -b]] = -2*a*b.
    Rend (matrice finale, nombre de papillons, facteur de determinant par papillon)."""
    a, b = scalaires
    # on part de la famille identite, dont le determinant vaut 1
    M = [[1 if i == j else 0 for j in range(N)] for i in range(N)]
    papillons = 0
    taille = 2
    while taille <= N:
        demi = taille // 2
        # un etage du reseau : N/taille blocs, chacun de demi papillons
        for base in range(0, N, taille):
            for j in range(demi):
                i1, i2 = base + j, base + j + demi
                u, v = M[i1], M[i2]
                # les deux formes du papillon, ecrites ligne par ligne
                M[i1] = [a * p + b * q for p, q in zip(u, v)]
                M[i2] = [a * p - b * q for p, q in zip(u, v)]
                papillons += 1
        taille *= 2
    return M, papillons, -2 * a * b


# 1) le determinant suivi papillon par papillon, contre celui de Bareiss
print("Argument de volume rejoue : le determinant, suivi papillon par papillon")
print()
print("      N    papillons    additions    determinant suivi    determinant Bareiss    |det| = N^(N/2)")
for N in [2, 4, 8, 16, 32]:
    M, papillons, facteur = rejouer_morgenstern(N)
    suivi = facteur ** papillons
    bareiss = determinant_bareiss(M)
    print(f"{N:7d} {papillons:12d} {2 * papillons:12d} {suivi:20d} {bareiss:22d} "
          f"{str(abs(suivi) == N ** (N // 2)):>18s}")

print()
print("La famille finale est exactement la matrice de Hadamard, de N = 2 a 32 :",
      all(rejouer_morgenstern(N)[0] == hadamard(N).tolist() for N in [2, 4, 8, 16, 32]))

print()
# 2) memes portes, coefficients libres : le volume ne borne plus le compte
print("Meme reseau, memes portes, autres coefficients (N = 16)")
print()
print("  scalaires    facteur par papillon    determinant suivi    determinant Bareiss    portes")
for scalaires in [(1, 1), (1, 2), (3, 3)]:
    M, papillons, facteur = rejouer_morgenstern(16, scalaires)
    suivi = facteur ** papillons
    bareiss = determinant_bareiss(M)
    print(f"{str(scalaires):>10s} {facteur:23d} {suivi:20d} {bareiss:22d} {2 * papillons:9d}")


Argument de volume rejoue : le determinant, suivi papillon par papillon

      N    papillons    additions    determinant suivi    determinant Bareiss    |det| = N^(N/2)
      2            1            2                   -2                     -2               True
      4            4            8                   16                     16               True
      8           12           24                 4096                   4096               True
     16           32           64           4294967296             4294967296               True
     32           80          160 1208925819614629174706176 1208925819614629174706176               True

La famille finale est exactement la matrice de Hadamard, de N = 2 a 32 : True

Meme reseau, memes portes, autres coefficients (N = 16)

  scalaires    facteur par papillon    determinant suivi    determinant Bareiss    portes
    (1, 1)                      -2           4294967296             4294967296        64
    (1, 2)           

### Lecture du résultat — l'argument tient par le module, et s'arrête au modèle borné

Le déterminant suivi papillon par papillon — signé — est **à chaque taille** celui que l'algorithme de Bareiss calcule indépendamment sur la matrice finale : $-2$, $16$, $4\,096$, $4\,294\,967\,296$, puis un nombre de 25 chiffres à $N = 32$. Et son module vaut $N^{N/2}$ : l'argument rejoué est bien celui du §1.2, et la vérification ne suppose rien de l'argument qu'elle contrôle. La famille finale est exactement la matrice de Hadamard. La colonne « additions » compte deux additions par papillon, soit $N\log_2 N$ pour la FFT — le double des $\frac{1}{2}N\log_2 N$ que la borne exige.

L'argument a une condition, et le second tableau la rend visible. **Relâcher la borne sur les coefficients suffit à tuer la mesure.** Avec les coefficients $(1, 2)$, chaque papillon multiplie le module du déterminant par $4$ au lieu de $2$ ; avec $(3, 3)$, par $18$. Au bout des mêmes $32$ papillons — des mêmes $64$ portes — le volume final compte $20$ chiffres, puis $41$, contre $10$ pour la FFT. Une seule porte peut donc multiplier le volume par un facteur arbitraire : **le déterminant n'est plus une mesure de complexité dès que les scalaires sont libres**, et c'est exactement le modèle du préprint.

| | Modèle à scalaires bornés (Morgenstern, 1973, cité) | Modèle du préprint (2026) |
|---|---|---|
| coefficients | de module au plus $1$ | quelconques dans $K$ |
| mesure disponible | module du déterminant, doublé au plus par addition | aucune mesure de volume : une porte le multiplie par ce qu'elle veut |
| borne inférieure | $\frac{1}{2}N\log_2 N$ additions | $c\,N\sqrt{\log N}$ portes |
| borne supérieure | $N\log_2 N$ additions (la FFT radix-2) | $C\,N(1+\log N)^{1/2+\varepsilon}$ portes |
| écart entre la borne inférieure et la FFT | un facteur $2$ | un facteur $\sqrt{\log N}$ par échantillon, à la constante $c$ près |

La borne inférieure du préprint ne peut donc plus venir du volume. Elle vient d'ailleurs, et le §1.3 le dit en une phrase : « Rank and the number of nonzero entries each describe one way a matrix can be easy to compute. Neither by itself explains the Fourier bound. » La preuve teste les deux ensemble — elle est **citée**, pas rejouée : ce carnet n'en implémente rien.

## 4. L'écart : $\sqrt{\log N}$, et pourquoi aucune taille où il se réalise ne peut être nommée

Les deux bornes du préprint s'écrivent par échantillon : $c\sqrt{k}$ en bas, $C(1+k)^{1/2+\varepsilon}$ en haut, où $k = \log_2 N$. La FFT, elle, coûte $k$ portes par échantillon. L'écart entre ce qu'on sait faire ($k$) et ce que la borne supérieure promet ($C(1+k)^{1/2+\varepsilon}$) est donc un facteur $k/(1+k)^{1/2+\varepsilon}$ : il **croît**, mais lentement, et il est commandé par une constante $C$ que le préprint ne donne pas.

La cellule suivante calcule trois choses, toutes par évaluation de formules — rien n'y est mesuré au sens expérimental : (1) le rapport $k/(1+k)^{1/2+\varepsilon}$ pour $\varepsilon \in \{0 ; 0{,}1 ; 0{,}25\}$ ; (2) la taille $k^{*}$ à partir de laquelle la borne supérieure descend sous le coût de la FFT, pour quelques valeurs de $C$ — c'est-à-dire la taille à partir de laquelle le circuit promis par la borne supérieure coûterait moins de portes que la FFT ; (3) l'exposant empirique de $k^{*}$ en fonction de $C$, à comparer à la prédiction $k^{*} \approx C^{1/(1/2-\varepsilon)}$ que donne l'inversion de $k \approx C\,k^{1/2+\varepsilon}$.

Le prédicat $k > C(1+k)^{1/2+\varepsilon}$ est **monotone** pour les $C \ge 1$ de la cellule : la fonction $f(k) = k - C(1+k)^p$ (avec $p = 1/2 + \varepsilon < 1$) est strictement négative en $k = 1$, puisque $C\,2^p > 1$ ; sa dérivée $1 - Cp(1+k)^{p-1}$ est croissante, donc $f$ décroît puis croît et ne s'annule qu'une fois. La dichotomie rend bien la plus petite taille, et non un point intermédiaire.

In [6]:
def croisement(C, eps, k_max=10 ** 9):
    """Plus petit k tel que k > C * (1 + k) ** (0.5 + eps), ou None si k_max ne suffit pas."""
    def au_dessus(k):
        return k > C * (1 + k) ** (0.5 + eps)

    # recherche exponentielle pour encadrer k*, puis dichotomie entre lo et hi
    lo, hi = 1, 2
    while hi <= k_max and not au_dessus(hi):
        lo, hi = hi, 2 * hi
    if hi > k_max:
        return None
    while lo < hi:
        milieu = (lo + hi) // 2
        if au_dessus(milieu):
            hi = milieu
        else:
            lo = milieu + 1
    return lo


# 1) l'ecart entre k (cote FFT) et (1+k)^alpha, pour trois exposants
print("Ecart entre les deux bornes, par echantillon (k = log2 N)")
print()
print("      k    FFT = k    (1+k)^0.5    rapport    (1+k)^0.6    rapport    (1+k)^0.75    rapport")
for k in [8, 16, 24, 32, 64, 128, 256, 512, 1024]:
    r0 = k / (1 + k) ** 0.5
    r1 = k / (1 + k) ** 0.6
    r2 = k / (1 + k) ** 0.75
    print(f"{k:7d} {k:11d} {math.sqrt(1 + k):12.3f} {r0:10.3f} {(1 + k) ** 0.6:12.3f} {r1:10.3f} "
          f"{(1 + k) ** 0.75:13.3f} {r2:10.3f}")

print()
# 2) la taille a laquelle la borne superieure bat la FFT, pour plusieurs C
print("Taille k* ou la borne superieure bat la FFT, et log10 du N* correspondant")
print()
for C in [1, 2, 4, 16, 64]:
    morceaux = []
    for eps in [0.0, 0.1, 0.25]:
        k_star = croisement(C, eps)
        if k_star is None:
            morceaux.append(f"eps={eps} : > 10^9          ")
        else:
            morceaux.append(f"eps={eps} : k*={k_star:>9d} (log10 N* = {k_star * math.log10(2):6.1f})")
    print(f"C={C:3d}  " + " | ".join(morceaux))

print()
# 3) l'exposant empirique : la pente de k* en fonction de C, contre 1/(1/2 - eps)
print("Exposant empirique ln(k*2/k*1) / ln(C2/C1), contre 1/(1/2 - eps)")
for eps in [0.0, 0.1, 0.25]:
    points = [(C, croisement(C, eps)) for C in [4, 16, 64]]
    for (C1, k1), (C2, k2) in zip(points, points[1:]):
        mesure = math.log(k2 / k1) / math.log(C2 / C1)
        print(f"eps={eps:<5} entre C={C1:2d} et C={C2:2d} : mesure {mesure:6.3f}   "
              f"theorie {1 / (0.5 - eps):6.3f}")


Ecart entre les deux bornes, par echantillon (k = log2 N)

      k    FFT = k    (1+k)^0.5    rapport    (1+k)^0.6    rapport    (1+k)^0.75    rapport
      8           8        3.000      2.667        3.737      2.141         5.196      1.540
     16          16        4.123      3.881        5.474      2.923         8.372      1.911
     24          24        5.000      4.800        6.899      3.479        11.180      2.147
     32          32        5.745      5.570        8.149      3.927        13.768      2.324
     64          64        8.062      7.938       12.239      5.229        22.892      2.796
    128         128       11.358     11.270       18.465      6.932        38.277      3.344
    256         256       16.031     15.969       27.923      9.168        64.187      3.988
    512         512       22.650     22.605       42.274     12.112       107.792      4.750
   1024        1024       32.016     31.984       64.037     15.991       181.152      5.653

Taille k* o

### Lecture du résultat — le gain est réel, lent, et commandé par la constante

Trois lectures, dans l'ordre des tableaux.

**Le rapport croît, mais très lentement.** À $k = 24$, le rapport vaut $4{,}8$ dans le cas le plus favorable des trois ($\varepsilon = 0$), $3{,}5$ avec $\varepsilon = 0{,}1$, $2{,}1$ avec $\varepsilon = 0{,}25$. À $k = 1\,024$ — $N$ à plus de 300 chiffres décimaux —, il vaut $32{,}0$, $16{,}0$ et $5{,}7$ : multiplier $N$ par $2^{1\,000}$ n'a même pas multiplié le gain par dix. Le gain calculé est un **facteur**, jamais un changement d'échelle.

**La constante décide de tout.** À $\varepsilon = 0$, avec $C = 1$, la borne bat la FFT dès $k = 2$ ; avec $C = 4$, il faut $k = 17$ ; avec $C = 16$, $k = 257$ ; avec $C = 64$, $k = 4\,097$ — où $N$ compte $1\,234$ chiffres décimaux. Le préprint ne donne **aucune** valeur de $C$ : le théorème dit seulement qu'elle dépend de $\varepsilon$. C'est une limite de l'énoncé lui-même, pas de sa lecture : sans $C$, aucune ligne de ce tableau ne dit à quelle taille le gain apparaît.

**L'exposant empirique suit la prédiction, et s'en approche quand $C$ grandit.** Sur les trois valeurs $C = 4$, $16$, $64$, le quotient de $k^{*}$ d'une constante à la suivante donne $1{,}959$ puis $1{,}997$ pour $\varepsilon = 0$ (prédiction $2$), $2{,}458$ puis $2{,}499$ pour $\varepsilon = 0{,}1$ (prédiction $2{,}5$), $3{,}992$ puis $4{,}000$ pour $\varepsilon = 0{,}25$ (prédiction $4$). La convergence vers $1/(1/2-\varepsilon)$ à mesure que $C$ grandit est un bon contrôle : la loi $k^{*} \approx C^{1/(1/2-\varepsilon)}$ est bien celle que le calcul suit, et elle rend visible la raison structurelle du problème — pour que le gain apparaisse tôt, il faut une constante petite, et pour $C$ grand la taille requise explose en $C^2$, $C^{2{,}5}$ ou $C^4$.

In [7]:
print("Temps d'une FFT complexe dans numpy -- une implementation, pas le modele en portes")
print()
debit = 0.0
for k in [12, 16, 20, 22]:
    N = 1 << k
    x = rng.standard_normal(N) + 1j * rng.standard_normal(N)
    np.fft.fft(x[:4096])                       # echauffement
    repetitions = 5
    depart = time.perf_counter()
    for _ in range(repetitions):
        np.fft.fft(x)
    duree = (time.perf_counter() - depart) / repetitions
    debit = (N * k) / duree
    print(f"k={k:2d}  N={N:9d}  t={duree * 1e3:9.3f} ms  t/N={duree / N * 1e9:7.2f} ns  portes/s={debit:.2e}")

print()
print("Nombre de portes de la FFT a grande taille -- des comptes exacts, independants de la machine")
print()


def puissance_de_deux(n: int) -> int:
    """Exposant e tel que n = 2^e, pour n entier puissance de 2."""
    exposant = 0
    while n > 1:
        n //= 2
        exposant += 1
    return exposant


print(f"{'k_ext':>7} {'portes = N*k':>14} {'log10(portes)':>16} {'depuis k = 64':>17}"
      f" {'depuis la ligne precedente':>28}")
precedent = None
premiere = None
for k_ext in [64, 128, 256]:
    portes = 2 ** k_ext * k_ext
    if premiere is None:
        premiere = portes
    depuis_64 = f"2^{puissance_de_deux(portes // premiere)}"
    depuis_precedent = "--" if precedent is None else f"2^{puissance_de_deux(portes // precedent)}"
    print(f"{k_ext:7d} {portes:>14.3e} {math.log10(portes):>16.1f} {depuis_64:>17} {depuis_precedent:>28}")
    precedent = portes

print()
print(f"Extrapolation : le debit mesure a k = 22 ({debit:.2e} portes/s), suppose constant jusqu'a ces tailles")
print()
for k_ext in [64, 128, 256]:
    portes = 2 ** k_ext * k_ext
    annees = portes / debit / 3.15576e7
    print(f"k_ext={k_ext:3d} : {annees:.3e} annees, soit 10^{math.log10(annees):.1f} annees,"
          f" {annees / 1.38e10:.3e} fois l'age de l'univers")

Temps d'une FFT complexe dans numpy -- une implementation, pas le modele en portes

k=12  N=     4096  t=    0.187 ms  t/N=  45.76 ns  portes/s=2.62e+08
k=16  N=    65536  t=    3.287 ms  t/N=  50.16 ns  portes/s=3.19e+08


k=20  N=  1048576  t=   70.195 ms  t/N=  66.94 ns  portes/s=2.99e+08


k=22  N=  4194304  t=  276.270 ms  t/N=  65.87 ns  portes/s=3.34e+08

Nombre de portes de la FFT a grande taille -- des comptes exacts, independants de la machine

  k_ext   portes = N*k    log10(portes)     depuis k = 64   depuis la ligne precedente
     64      1.181e+21             21.1               2^0                           --
    128      4.356e+40             40.6              2^65                         2^65
    256      2.964e+79             79.5             2^194                        2^129

Extrapolation : le debit mesure a k = 22 (3.34e+08 portes/s), suppose constant jusqu'a ces tailles

k_ext= 64 : 1.120e+05 annees, soit 10^5.0 annees, 8.116e-06 fois l'age de l'univers
k_ext=128 : 4.132e+24 annees, soit 10^24.6 annees, 2.994e+14 fois l'age de l'univers
k_ext=256 : 2.812e+63 annees, soit 10^63.4 annees, 2.038e+53 fois l'age de l'univers


### Lecture du résultat — un temps ne mesure pas des portes

Le temps par échantillon mesuré **ne se déduit pas de $k$**. D'un bout à l'autre de la plage, $k$ augmente de moins d'un facteur 2 ; le débit affiché, en portes par seconde, n'est lui **pas constant** — il change d'une taille à l'autre, sans rapport fixe avec le compte de portes. L'implémentation paie sans doute autre chose que les portes — accès mémoire, vectorisation — ; ce carnet ne le mesure pas, et le modèle du préprint, lui, ne facture que les portes. Le préprint donne la traduction qu'il autorise : « Conversely, three field operations suffice for a weighted linear gate. » Un compte de portes se traduit en **opérations**, au plus trois par porte ; il ne se traduit pas en temps sans un modèle de machine.

Ce que la cellule sert à dire, en revanche, tient en deux faits de nature différente. Le premier est exact et indépendant de la machine : **les facteurs entre tailles sont exacts** — passer de $k = 64$ à $k = 128$ multiplie le nombre de portes de la FFT par $2^{65}$, et de $k = 128$ à $k = 256$ par $2^{129}$. Le second est une **extrapolation** : les trois dernières lignes prolongent jusqu'à ces tailles le débit mesuré à $k = 22$, en supposant qu'il y tienne — ce que la première moitié de la sortie donne des raisons de trouver optimiste, puisqu'il ne tient déjà pas entre $k = 12$ et $k = 22$. Dans cette extrapolation, le gain de la section 4 — un facteur $16$ à $k = 256$ dans le cas le plus favorable des trois ($\varepsilon = 0$) — n'abaisse le logarithme décimal de la durée que d'un peu plus d'une unité. Le théorème améliore l'**exposant**, pas la **faisabilité** ; et ces durées sont celles d'une machine réelle, pas celles du modèle.

## 5. La borne inférieure comme objet

Une borne inférieure n'est pas une mesure, c'est un **quantificateur** : $C_K(T) \ge c\,J\,N\sqrt{\log N}$ affirme que **tout** circuit acyclique qui calcule $T$ paie au moins cela — quelle que soit l'astuce, quels que soient les coefficients. La conséquence méthodologique est asymétrique, et c'est le point le plus important de ce carnet :

- **une borne supérieure se démontre en exhibant un circuit** — c'est ce que fait la section 2 pour la FFT, avec $N\log_2 N$ portes, et c'est ce qu'un ordinateur peut contrôler ;
- **une borne inférieure ne se mesure pas** : aucune expérience ne parcourt tous les circuits. Elle ne se réfute que par un seul contre-exemple — un circuit qui descendrait sous la borne — et c'est pourquoi les bornes inférieures se publient avec une preuve, jamais avec un banc d'essai.

Le préprint renforce même l'énoncé (Théorème 1.2, « Robust lower bound ») : avec $h = JN$ et des constantes absolues $c, \rho > 0$, si $T$ appartient à l'adhérence de Zariski des sommes $M + B + Z$, où $M$ est calculé par un système d'au plus $s$ états à rétroaction, $\operatorname{rank} B \le \rho\,h$ et $|\operatorname{supp} Z| \le h\sqrt{2\log N}/8$, alors $s \ge c\,h\sqrt{\log N}$. Autrement dit la borne survit à la rétroaction, à une erreur de rang faible et à une erreur creuse prises ensemble ; et le préprint ajoute : « Setting the corrections to zero and using an acyclic circuit gives the lower bound of Theorem 1.1. »

Ce que la mesure peut faire, à défaut de vérifier la borne, est un **contrôle de cohérence** : la borne inférieure, appliquée aux plus petites tailles, ne doit pas contredire les circuits qu'on vient d'exhiber. La cellule suivante fait ce contrôle — en prenant $J = 1$ et en posant la constante $c$ égale à $1$ — et en tire la seule conclusion mesurable de cette section : une inégalité sur $c$.

In [8]:
# la borne inferieure N*sqrt(k) avec c = 1, contre les portes reellement comptees
print("Controle de coherence : la borne inferieure avec c = 1, contre les circuits de la section 2")
print()
print("      N    portes du circuit    borne avec c = 1 : N*sqrt(k)    marge")
marges = []
for k in range(1, 11):
    N = 1 << k
    n_portes = len(programme_wht(N)[0])
    basse = N * math.sqrt(k)
    marges.append((n_portes / basse, N, n_portes))
    print(f"{N:7d} {n_portes:19d} {basse:28.3f} {n_portes / basse:9.3f}")

# la plus petite marge sur la plage fixe la plus grande constante c compatible
marge_min, n_min, portes_min = min(marges)
print()
print(f"Plus petite marge sur la plage : {marge_min:.3f}, atteinte a N = {n_min} ({portes_min} portes).")
print(f"Plus grande constante c compatible avec ce circuit : c <= {portes_min / (n_min * math.sqrt(math.log2(n_min))):.3f}")


Controle de coherence : la borne inferieure avec c = 1, contre les circuits de la section 2

      N    portes du circuit    borne avec c = 1 : N*sqrt(k)    marge
      2                   2                        2.000     1.000
      4                   8                        5.657     1.414
      8                  24                       13.856     1.732
     16                  64                       32.000     2.000
     32                 160                       71.554     2.236
     64                 384                      156.767     2.449
    128                 896                      338.656     2.646
    256                2048                      724.077     2.828
    512                4608                     1536.000     3.000
   1024               10240                     3238.172     3.162

Plus petite marge sur la plage : 1.000, atteinte a N = 2 (2 portes).
Plus grande constante c compatible avec ce circuit : c <= 1.000


### Lecture du résultat — la marge vaut $\sqrt{k}$, et c'est l'écart de la section 4

À $N = 2$, le circuit à deux portes de la section 2 et la borne inférieure se touchent : la marge vaut exactement $1$, et l'inégalité $c \cdot 2\sqrt{1} \le 2$ impose $c \le 1$. C'est la seule constante que ce carnet peut fixer, et elle vient d'un circuit, pas d'une mesure.

Partout ailleurs, la marge entre le circuit exhibé ($Nk$ portes) et la borne ($N\sqrt{k}$) vaut exactement $\sqrt{k}$ — la colonne du tableau, qui retrouve le rapport de la section 4 sans avoir été calculée de la même façon. C'est le contrôle attendu : les deux sections lisent le même écart, l'une par le haut (ce que promet la borne supérieure), l'autre par le bas (ce que garantit la borne inférieure). Il n'y a pas de contradiction, et il n'y a pas de démonstration non plus : la borne inférieure reste **citée**, elle n'est ni rejouée ni vérifiée ici. Tout ce que ce carnet peut en dire est qu'elle est cohérente avec les circuits qu'il écrit, et que son énoncé porte sur tous les circuits — ce qu'aucune cellule ne parcourt.

## 6. Statut de la formalisation, tel que trouvé

Le PDF ne dit **rien** du Lean : ni « Lean » ni « formaliz » n'y apparaissent une seule fois (relevé sur le texte extrait du fichier archivé au gisement, le 26 novembre 2026). Le statut de formalisation ne se lit donc pas dans l'article — il se lit **à côté**, dans le dossier qui le publie, et c'est là que ce carnet le prend, tel qu'il est écrit.

### Ce que le dossier du preprint déclare

Le dossier du preprint — `preprints/The-logarithmic-exponent-of-finite-Fourier-transforms-November-24-2026/` — porte une édition datée, « OpenAI · November 24, 2026 edition », et une section « Lean formalization » dont voici la phrase entière :

> The Lean development includes the upper and lower bounds for individual Fourier transforms and for batches of transforms.

Elle renvoie à trois emplacements du dépôt `openai/math`, écrits ici comme des chemins et non comme des liens, puisqu'ils vivent dans un autre dépôt que celui-ci : `lean/OAI/LinearAlgebra/FiniteFourier/Main.lean` (lien « Single-transform proof », ligne 17), `lean/OAI/LinearAlgebra/FiniteFourier/BatchedTheorem.lean` (lien « Batched-transform proof », ligne 93), et `lean/docs/130.md` (lien « Formalization scope »).

### La portée, mot pour mot

`lean/docs/130.md` s'intitule « Exact Fourier transforms, batched circuits, and uniform algorithms » et couvre **trois** préprints : *Finite tensor savings and exact Fourier circuits* et *An explicit power saving for the exact discrete Fourier transform* (tous deux datés du 25 septembre 2026), puis celui-ci. Le paragraphe de portée, pour une transformée seule :

> For a finite abelian group of order $n\ge2$, the forward and normalized inverse Fourier transforms each require $\Omega(n\sqrt{\log n})$ binary linear gates and admit circuits of size $O_\varepsilon(n(\log n)^{1/2+\varepsilon})$ for every $\varepsilon>0$. Each gate forms a weighted sum of two wires. The field must contain a primitive root of unity of order equal to the group exponent, and $n$ must be invertible. This includes the complex discrete Fourier transform (DFT) at every $n\ge2$.

et pour le lot de transformées :

> For $J\ge1$ transforms of the same order over a common admissible field, both bounds scale by $J$, even with different groups, mixed directions, and gates shared across transforms. The lower bound also holds for feedback states under Zariski limits with simultaneous low-rank and sparse errors across the batch.

Le même document nomme, dans une table « Comparator links », les quatre **défis de comparaison** que le développement doit relever. Trois d'entre eux renvoient au fichier `ComparatorChallenges/ExactFourier.lean` : « Forward and inverse transforms, upper and lower circuit bounds » (lignes 232 à 245), « Batched transforms, upper and lower circuit bounds » (lignes 294 à 310) et « Subsequential savings for exact complex DFT circuits » (lignes 51 à 57) ; le quatrième, « Uniform exact DFT and convolution with a power saving », renvoie à `ComparatorChallenges/UniformFourier.lean` (lignes 317 à 330).

### Un écart de vocabulaire, relevé et non interprété

La portée écrit « the forward and **normalized** inverse Fourier transforms » ; le théorème de l'article écrit « choosing the forward or inverse transform independently in each block ». Les deux phrases ne disent pas la même chose de l'inverse — normalisée d'un côté, au choix de l'autre — et ce carnet ne tranche pas. Il rappelle seulement ce que le modèle facture, cité en section 1 : « An output may select and scale a single input or gate value » — mettre à l'échelle une valeur au moment de la lire ne coûte aucune porte. Ce que la formalisation, elle, a retenu de plus ne se lit que dans ses fichiers.

### Ce que ce carnet ne fait pas

Aucun `lake build` n'est lancé ici : sur ce poste, un lac complet dépasse la mémoire disponible, et la compilation de preuve appartient aux runners. Les chemins ci-dessus sont donc **rapportés** — lus dans les fichiers qui les nomment, le 26 novembre 2026 — et non vérifiés par une exécution. C'est le niveau **Cité** de la section suivante : un développement Lean qui se déclare à ces emplacements, pour un article qui ne le mentionne pas.

## 7. Mesuré, cité, ouvert

Le carnet tient trois registres séparés : ce qu'il a **exécuté**, ce qu'il a **lu** dans une source, et ce que **personne n'a établi** à ce jour. Le théorème du papier est dans le deuxième — aucun des circuits qui font tenir ses deux bornes n'est exécuté ici.

| Niveau | Ce que le carnet en dit | Où |
|---|---|---|
| **Mesuré** | Le réseau papillon radix-2 de la transformée de Walsh-Hadamard coûte exactement $N\log_2 N$ portes de $N = 2$ à $1\,024$ ; sa matrice est exactement $H_N$ et sa valeur exacte en arithmétique entière. Le même réseau, la racine de l'unité passée dans le coefficient des portes, rend la transformée de Fourier complexe à la précision de la machine et sa matrice, à $N = 8$, est celle de Fourier. Une matrice **dense** et inversible, $I + J$, se calcule en $2N - 1$ portes. Dans le réseau papillon, le déterminant de la famille est multiplié par $-2ab$ à chaque papillon : le produit suivi, signe compris, est le déterminant calculé par Bareiss sur la matrice finale, et vaut $N^{N/2}$ en module ; avec $a = b = 3$, il croît d'un facteur 18 par papillon. L'écart entre les deux bornes, la taille du croisement et son exposant empirique se calculent. La borne inférieure prise avec $c = 1$ laisse sur tous les circuits mesurés une marge de $\sqrt{k}$, minimale en $N = 2$ où elle vaut 1 : $c \le 1$ pour ce théorème. | §2, §3, §4, §5 |
| **Cité** | Toute la borne inférieure et sa preuve : rang et support testés ensemble sur un rectangle, l'argument de volume en modèle borné, la clôture de Zariski et le théorème robuste 1.2. La borne supérieure $C\,J\,N(1+\log N)^{1/2+\varepsilon}$ : aucun circuit du papier n'est implémenté ici, et **aucune valeur de $C$ n'est donnée**. Le statut de la formalisation, tel que son dossier le déclare. Les travaux antérieurs de la section 1.2 du papier, cités et non lus. | §1, §3, §5, §6 |
| **Ouvert** | Une borne supérieure avec $\varepsilon = 0$ — le papier l'écrit lui-même. La valeur de $C$. Un algorithme dont la constante serait connue, donc un circuit qui batte la FFT à une taille où l'on pourrait l'exécuter. | §1, §4 |

Un mot sur la dernière ligne du registre **Mesuré** : ce que le contrôle établit est qu'un théorème à constante inconnue ne contredit aucun des circuits comptés. Ce n'est pas une vérification du théorème, et le carnet ne la présente pas comme telle.

## 8. Exercices

Trois exercices, sur le modèle de la série : les fonctions sont données en stubs non bloquants, et le carnet s'exécute de bout en bout même non complété. Chaque énoncé dit ce que la fonction doit rendre ; les indices mènent à la solution, pas à une solution.

### Exercice 1 — un réseau radix-4 coûte-t-il moins de portes ?

Le réseau de la section 2 est **radix-2** : chaque papillon prend deux valeurs et coûte deux portes. Un papillon **radix-4** prend quatre valeurs $(u, v, w, z)$ et rend les quatre sommes de Hadamard, c'est-à-dire $H_4\,(u, v, w, z)^{\mathsf T}$. Avec des portes binaires, il se décompose en huit : deux pour $(u + v,\ u - v)$, deux pour $(w + z,\ w - z)$, puis quatre pour $(u + v) \pm (w + z)$ et $(u - v) \pm (w - z)$. Un réseau radix-4 de taille $N = 4^m$ applique $m$ étages de $N/4$ papillons de ce type.

Écrire `compter_radix4(N)`, qui prend $N = 4^m$ et rend le nombre total de portes de ce réseau, un entier. La cellule l'appelle pour $N = 16$, à côté du compte radix-2 de la section 2 — comparez.

In [9]:
def compter_radix4(N):
    """Exercice 1. Nombre de portes binaires d'un reseau radix-4 de taille N = 4^m.

    Un papillon radix-4 se decompose en 8 portes binaires (enonce ci-dessus).
    Un reseau de taille N applique m = log4(N) etages de N/4 papillons.
    Retour attendu : un entier, le nombre total de portes.
    """
    # TODO etudiant : multiplier le nombre de papillons par le cout d'un papillon
    # Indice : m = ? etages, N/4 papillons par etage, 8 portes par papillon
    # Etape 1 : compter les etages (attention, log4 n'est pas log2)
    # Etape 2 : assembler le produit et le rendre
    print("Exercice 1 : compter_radix4 definie -- a completer (retour attendu : un entier).")
    return None


N_4 = 16
print(f"A N = {N_4} : radix-4 =", compter_radix4(N_4),
      f"| radix-2 mesure en section 2 = {len(programme_wht(N_4)[0])} | N*log2(N) = {N_4 * 4}")

Exercice 1 : compter_radix4 definie -- a completer (retour attendu : un entier).
A N = 16 : radix-4 = None | radix-2 mesure en section 2 = 64 | N*log2(N) = 64


### Exercice 2 — l'argument de volume survit-il à la normalisation ?

Morgenstern compte les additions parce que le déterminant **grandit** : dans le réseau papillon, chaque papillon le multiplie par $-2ab$, et le produit final vaut $N^{N/2}$. Le papier note la parade, en une phrase de la section 1.2 : « Normalizing the transform makes its determinant have modulus one. » Diviser chaque sortie par $\sqrt{N}$ divise le déterminant par $(\sqrt{N})^{N} = N^{N/2}$ — le module tombe à 1, et un argument qui compte les doublements n'a plus rien à compter.

Écrire `module_determinant_normalise(N)`, qui rend le module du déterminant de $H_N/\sqrt{N}$, **en exact** : $\left|\det H_N\right| / N^{N/2}$, calculé avec `determinant_bareiss` (un entier ou un `Fraction`, jamais un flottant, qui ne saurait rendre un 1 exact). Elle doit rendre `1`. La cellule l'appelle pour $N = 2, 4, 8, 16$ ; et la question à garder en tête est celle de la section 5 : la borne inférieure du papier vaut **malgré** ce module de 1 — comment un argument de volume peut-il tomber et la borne tenir ?

In [10]:
def module_determinant_normalise(N):
    """Exercice 2. Module du determinant de H_N / sqrt(N), calcule en exact.

    det(H_N / sqrt(N)) = det(H_N) / (sqrt(N))**N = det(H_N) / N**(N/2).
    Retour attendu : 1 (un entier, ou une Fraction egale a 1).
    """
    # TODO etudiant : determiner le module de det(H_N), puis diviser par N**(N/2)
    # Indice : determinant_bareiss rend le determinant SIGNE ; le module est sa valeur absolue
    # Etape 1 : |det(H_N)| par Bareiss, sur hadamard(N)
    # Etape 2 : diviser par N**(N/2) -- attention, N/2 est un entier ici
    return None


for N in [2, 4, 8, 16]:
    print(f"N = {N:2d} : |det(H_N / sqrt(N))| = {module_determinant_normalise(N)}")

N =  2 : |det(H_N / sqrt(N))| = None
N =  4 : |det(H_N / sqrt(N))| = None
N =  8 : |det(H_N / sqrt(N))| = None
N = 16 : |det(H_N / sqrt(N))| = None


### Exercice 3 — mesurer l'exposant du croisement

La section 4 a montré que la taille $k^*$ à laquelle la borne supérieure bat la FFT croît comme $C^{1/(1/2-\varepsilon)}$ : l'exposant **empirique** d'une paire de constantes est $\ln(k^*_2/k^*_1) / \ln(C_2/C_1)$, et il doit tendre vers $1/(1/2-\varepsilon)$ quand la constante grandit. La constante $C$ du théorème étant inconnue, cet exposant est tout ce que l'on peut mesurer de la croissance.

Écrire `exposant_empirique(eps)`, qui prend $\varepsilon$ et rend la liste des deux exposants mesurés entre les constantes $C = 4$, $16$ et $64$ (dans cet ordre), en utilisant `croisement` telle que définie en section 4. Elle rend une liste de deux flottants. La cellule l'appelle pour $\varepsilon = 0$, $0{,}1$ et $0{,}25$, à côté de la valeur théorique — vérifiez que la mesure suit la théorie quand $C$ grandit, et dites à quel point elle s'en approche.

In [11]:
def exposant_empirique(eps):
    """Exercice 3. Exposants mesures du croisement, entre C = 4, 16 et 64.

    croisement(C, eps) (section 4) rend k*(C, eps). L'exposant mesure entre deux
    constantes C1 < C2 vaut ln(k*(C2) / k*(C1)) / ln(C2 / C1).
    Retour attendu : la liste des deux exposants, pour les paires (4, 16) puis (16, 64).
    """
    # TODO etudiant : calculer k* pour C = 4, 16, 64, puis les deux quotients en logarithme
    # Indice : math.log est le logarithme neperien ; la base se simplifie dans le quotient
    # Etape 1 : la liste des k* pour les trois constantes
    # Etape 2 : un quotient par paire consecutive (la base du logarithme s'annule)
    return None

for eps in [0.0, 0.1, 0.25]:
    print(f"eps={eps:<5} mesure = {exposant_empirique(eps)}   theorie = {1 / (0.5 - eps):.3f}")

eps=0.0   mesure = None   theorie = 2.000
eps=0.1   mesure = None   theorie = 2.500
eps=0.25  mesure = None   theorie = 4.000


## 9. Conclusion

| Geste | Où | Ce qui a été fait | Niveau |
|---|---|---|---|
| Le modèle, mot pour mot | §1 | Une porte calcule $au + bv$ ; sélection, mise à l'échelle et racine de l'unité ne coûtent rien ; aucune constante explicite | Cité |
| Le compte de la FFT | §2 | $N\log_2 N$ portes, exactement, de $N = 2$ à $1\,024$ ; matrice $= H_N$ ; réseau complexe d'accord avec `numpy.fft.fft` | **Mesuré** |
| Dense mais linéaire | §2 | $I + J$ : $2N - 1$ portes, déterminant $N + 1$ | **Mesuré** |
| Morgenstern rejoué | §3 | Déterminant suivi, signe compris, égal à Bareiss ; $N^{N/2}$ ; facteurs 4 et 18 avec d'autres scalaires, mêmes portes | **Mesuré** (le rejoue), Cité (l'énoncé) |
| L'écart et son croisement | §4, §5 | Marge $\sqrt{k}$ ; $k^* = 2,\ 17,\ 257,\ 4\,097$ pour $C = 1,\ 4,\ 16,\ 64$ à $\varepsilon = 0$ ; exposant empirique $1{,}959 \to 1{,}997$ pour une théorie de 2 | **Mesuré** (évaluation de formules) |
| Un temps n'est pas un compte | §4 | Chronomètre d'une implémentation, facteurs exacts $2^{65}$ et $2^{129}$ entre les tailles | **Mesuré** (temps) et exact (facteurs) |
| La borne inférieure | §5 | Lue, paraphrasée, jamais rejouée ; contrôle de cohérence : $c \le 1$ | Cité, et un contrôle **Mesuré** |
| La formalisation | §6 | README du dossier et `lean/docs/130.md`, tels qu'écrits ; aucun fichier `.lean` ouvert | Cité |

La conclusion en une phrase : **la transformée de Fourier de $N$ points se calcule avec $N\log_2 N$ portes — exactement, et c'est mesuré — alors qu'un théorème de novembre 2026 l'encadre entre $c\,N\sqrt{\log N}$ et $C\,N(\log N)^{1/2+\varepsilon}$ : un gain d'exposant, lu et non rejoué, dont aucune taille où il battrait la FFT ne peut être nommée tant que $C$ reste inconnu.**

### Références (copie archivée au gisement)

- **Préprint** — OpenAI, *The logarithmic exponent of finite Fourier transforms*, OpenAI Math Release preprint, édition du 24 novembre 2026, identifiant `OAI:The-logarithmic-exponent-of-finite-Fourier-transforms-November-24-2026` : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\2026 - OpenAI - The logarithmic exponent of finite Fourier transforms (openai-math preprint, 24 Nov 2026).pdf`
- **Statut de formalisation** — le `README.md` du dossier du préprint et `lean/docs/130.md`, dans le dépôt `openai/math`, lus le 26 novembre 2026 ; aucun fichier `.lean` n'a été ouvert et aucun `lake build` lancé.
- **Cités d'après la bibliographie et la section 1.2 du préprint, originaux non lus** — James W. Cooley & John W. Tukey, *An algorithm for the machine calculation of complex Fourier series*, Mathematics of Computation 19(90):297-301, 1965 ; Jacques Morgenstern, *Note on a lower bound on the linear complexity of the fast Fourier transform*, Journal of the ACM 20(2):305-306, 1973.
- **Dans la série** : [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) · [03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) · [03e — barrières d'exposants, word RAM](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) · [03f — la chaîne des réductions](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb)

*Issue de suivi : #28733. Carnet écrit le 26 novembre 2026 ; les faits datés (préprint, statut de formalisation) sont ceux de cette date.*

## À retenir

1. **Le modèle ne facture que les portes.** Chaque porte forme $au + bv$ ; les scalaires, les twiddles, la lecture des sorties et l'indexation ne coûtent rien. Dans ce modèle, la FFT radix-2 coûte exactement $N\log_2 N$ portes — 2 à $N = 2$, $10\,240$ à $N = 1\,024$ — contre $N(N-1)$ pour le produit matrice-vecteur, soit $1\,047\,552$ à $N = 1\,024$ : un facteur $102{,}3$, **mesuré**.

2. **La densité n'est pas la difficulté.** $I + J$ est dense, inversible ($\det = N + 1$) et se calcule en $2N - 1$ portes (**mesuré**). Ni le rang ni le nombre d'entrées non nulles n'expliquent seuls la borne de Fourier — c'est ce qu'écrit la §1.3 du préprint, et la preuve de la borne inférieure teste rang et support **ensemble** (**cité**).

3. **L'argument de volume s'arrête au modèle borné.** Dans le réseau papillon, chaque papillon multiplie le déterminant par $-2ab$ (**mesuré**). La borne de Morgenstern — $\frac{1}{2}N\log_2 N$ additions quand les scalaires sont de module au plus 1 — laisse donc à la FFT un facteur 2 ; avec $(a, b) = (1, 2)$ ou $(3, 3)$, les mêmes 64 portes multiplient le déterminant par 4 ou 18 par papillon, soit 20 ou 41 chiffres contre 10. Le théorème de 2026 autorise ces coefficients : « Theorem 1.1 permits arbitrary scalar coefficients, intermediate storage and reuse. » (**cité**).

4. **L'encadrement est un exposant, pas un algorithme.** Les deux bornes partagent l'exposant logarithmique $\tfrac{1}{2} + o(1)$ (**cité**). Entre la FFT, en $N\log_2 N$ portes, et la borne inférieure, en $c\,N\sqrt{\log N}$, l'écart par échantillon est donc un facteur $\sqrt{\log N}$, à la constante $c$ près : c'est le gain maximal que le théorème laisse à un circuit. Le croisement de la FFT avec la borne supérieure se calcule : $k^{*} = 17$ pour $C = 4$, $4\,097$ pour $C = 64$ à $\varepsilon = 0$ — cette dernière taille compte $1\,234$ chiffres décimaux (**mesuré**). Sans valeur de $C$, aucune taille où le gain se réalise ne peut être nommée (**ouvert**).

## Vérifiez votre compréhension

1. La FFT radix-2 compte $\frac{N}{2}\log_2 N$ papillons. Pourquoi le compte mesuré est-il $N\log_2 N$ portes, et non $\frac{N}{2}\log_2 N$ ni $2N\log_2 N$ ?
2. L'argument de Morgenstern donne $\frac{1}{2}N\log_2 N$ additions. Sur quelle hypothèse porte-t-il, et pourquoi ne suffit-elle plus dans le modèle du préprint ?
3. La borne supérieure promet un circuit de $C\,N(1+\log N)^{1/2+\varepsilon}$ portes. Pourquoi ne rend-elle pas calculable une FFT de $2^{64}$ échantillons ?

<details><summary>Réponses</summary>

1. Un papillon $(u, v) \mapsto (u + v,\; u - v)$ est **deux** portes : l'une de coefficients $(1, 1)$, l'autre de coefficients $(1, -1)$. Une porte forme une seule valeur, $au + bv$ ; elle ne peut pas rendre les deux sorties du papillon. D'où $2 \times \frac{N}{2}\log_2 N = N\log_2 N$ — et pas $2N\log_2 N$, car les multiplications par les racines de l'unité et la lecture des sorties ne sont pas facturées.
2. Il suppose que les scalaires sont de module au plus 1 : une addition ne peut alors que doubler le plus grand module de déterminant atteignable. Le préprint autorise au contraire des coefficients arbitraires. Le rejouer avec $(a, b) = (3, 3)$ le montre : les mêmes 64 portes multiplient le déterminant par 18 à chaque papillon au lieu de 2.
3. Parce que la borne est asymptotique et que sa constante $C$ n'est pas donnée. Le croisement avec la FFT croît comme $C^{2}$ pour $\varepsilon = 0$ : il vaut $k^{*} = 4\,097$ dès $C = 64$, taille où $N$ compte $1\,234$ chiffres décimaux. Et même croisé, le gain est un **facteur** — au plus $\sqrt{k}$ — quand la FFT de $2^{64}$ échantillons coûte déjà de l'ordre de $10^{21}$ portes : un facteur n'est pas un changement d'échelle.

</details>

## Pour aller plus loin

- **Le frère de la même journée** : l'autre préprint du 24 novembre 2026, *The matrix multiplication exponent is nine fourths*, ferme l'exposant du produit matriciel là où celui-ci encadre celui de la transformée de Fourier. Il est lu dans le carnet 03h de cette série (en revue sous #28554 au jour de l'écriture, donc non lié ici). Comparer les deux est un exercice en soi : même forme d'énoncé, deux statuts différents — un exposant **clos** d'un côté, un encadrement à exposant **partagé** de l'autre, avec $\varepsilon = 0$ toujours ouvert.
- **Dans la même série** : [03e — barrières d'exposants, word RAM](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) — le même geste, sur une autre source : lire un modèle de calcul à la machine avant de croire un exposant ; [03f — la chaîne des réductions, du 3SUM au triangle lopsided](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb) — comment un gain d'exposant se propage.
- **Carnet voisin d'une autre série** : [Z3-20 — La machine de Gödel (Python)](../../SymbolicAI/SMT/Z3-API/Z3-20-Machine-de-Goedel-Python.ipynb) — un programme qui ne se réécrit que sur preuve, le temps de preuve étant compté dans l'utilité : une autre façon de faire payer un programme, par ce qu'il prouve plutôt que par ce qu'il calcule.
- **Lectures** : le préprint d'OpenAI (24 novembre 2026) — copie archivée au gisement partagé, chemin complet dans les Références de la Conclusion ci-dessus. Jacques Morgenstern (1973) et James Cooley & John Tukey (1965) y sont cités : leurs originaux n'ont **pas** été lus pour écrire ce carnet, et leurs résultats y sont rapportés d'après la section 1.2 du préprint ; les autres travaux de cette section (Ailon, l'algorithme de [34, Theorem 1.1]) ne le sont que par leur mention.